<a href="https://colab.research.google.com/github/imanuni/imanuni/blob/main/nettoyagefile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade camel-tools[all]

In [14]:
!pip install --upgrade --force-reinstall numpy

  Using cached numpy-2.3.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (62 kB)
Using cached numpy-2.3.1-cp311-cp311-manylinux_2_28_x86_64.whl (16.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.1
    Uninstalling numpy-2.3.1:
      Successfully uninstalled numpy-2.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
camel-tools 1.5.6 requires numpy<2, but you have numpy 2.3.1 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.3.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.1 which is incompatible.
cupy-cuda12x 13.3.0 requires numpy<2.3,>=1.22, but you have numpy 2.3.1 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.1 which is incompatible.
opencv-python 4.12.0.88 re

In [1]:
import pandas as pd
import re

from camel_tools.utils.dediac import dediac_ar

# Nettoyage manuel

def clean_custom(text):
    # Supprimer les diacritiques (facultatif)
    text = re.sub(r'[\u064B-\u0652]', '', text)

    # Supprimer tous les chiffres (occidentaux et arabes)
    text = re.sub(r'[\d\u0660-\u0669]+', '', text)
     # Supprimer ponctuation arabe manuellement
    text = re.sub(r'[؟،؛]', '', text)  # <- ici on enlève ؟ et ses amis

    # Supprimer les ponctuations et symboles (mais garder les espaces et les deux points)
    text = re.sub(r'[^\u0600-\u06FF\s:]', '', text)

    # Réduire les espaces multiples à un seul
    text = re.sub(r'\s+', ' ', text)

    # Supprimer les espaces en début et fin
    return text.strip()


# 🔽 Télécharger et lire le fichier
from google.colab import files
uploaded = files.upload()

# Lire le fichier CSV
df = pd.read_csv('عناوين نهائي.csv', encoding='utf-8-sig')

# Nettoyage de la colonne "النص"
df['clean_text'] = df['النص'].astype(str).apply(clean_custom)

# Sauvegarde dans un fichier temporaire
df.to_csv('nettoyer.csv', index=False, encoding='utf-8-sig')

# 📤 Télécharger le fichier nettoyé
files.download('nettoyer.csv')

df.head()


Saving عناوين نهائي.csv to عناوين نهائي.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,النص,yes / no,clean_text
0,– عون: لحكومة تمثل الجميع… مطالب كتل تؤخّر الت...,yes,عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف
1,– زيارة إيرانية للراعي: رسالة وتطمينات وسجادة ...,no,زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بق...
2,– تفاؤل بحلّ قضية الودائع: على أي أسس؟!,no,تفاؤل بحل قضية الودائع: على أي أسس
3,– بوادر حملة عسكرية إسرائيلية – غربية: صنعاء ت...,no,بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر...
4,– وزير الخارجية السعودي إلى بيروت بين «الخميسي...,no,وزير الخارجية السعودي إلى بيروت بين الخميسين و...


الكود الموجود في الاسفل نجح في دمج : مع الاسم الذي يسبقها

In [2]:
import pandas as pd
import re
from google.colab import files

# 🔽 Upload fichier
uploaded = files.upload()

# 🔽 Lire fichier CSV
df = pd.read_csv('nettoyer.csv', encoding='utf-8-sig')
phrases = df['النص'].astype(str).tolist()

# 🔽 Fonction de normalisation arabe
def normalize_arabic(text):
    text = re.sub(r'[^\u0600-\u06FF\s:؛]', '', text)  # garder lettres arabes + : + ؛
    text = re.sub(r'[\u064B-\u0652]', '', text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    return text

# 🔽 Coller ":"  au mot précédent (supprime espace avant)
def attach_punct_to_previous_word(text):
    return re.sub(r'\s([:])', r'\1', text)

# 🔽 Tokenisation simple par split sur espace
def simple_tokenize(text):
    return text.split()

# 🔽 Traitement complet
tokens_column = []
for phrase in phrases:
    cleaned = normalize_arabic(phrase)
    cleaned = attach_punct_to_previous_word(cleaned)
    tokens = simple_tokenize(cleaned)
    tokens_column.append(tokens)

# 🔽 Ajouter les tokens dans une nouvelle colonne
df['tokens'] = tokens_column

# 🔽 Sauvegarde et téléchargement
df.to_csv('nettoye_tokens.csv', index=False, encoding='utf-8-sig')
files.download('nettoye_tokens.csv')

df.head()

Saving nettoyer.csv to nettoyer (1).csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,النص,yes / no,clean_text,tokens
0,– عون: لحكومة تمثل الجميع… مطالب كتل تؤخّر الت...,yes,عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف,"[عون:, لحكومه, تمثل, الجميع, مطالب, كتل, توخر,..."
1,– زيارة إيرانية للراعي: رسالة وتطمينات وسجادة ...,no,زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بق...,"[زياره, ايرانيه, للراعي:, رساله, وتطمينات, وسج..."
2,– تفاؤل بحلّ قضية الودائع: على أي أسس؟!,no,تفاؤل بحل قضية الودائع: على أي أسس,"[تفاول, بحل, قضيه, الودايع:, علي, اي, اسس؟]"
3,– بوادر حملة عسكرية إسرائيلية – غربية: صنعاء ت...,no,بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر...,"[بوادر, حمله, عسكريه, اسراييليه, غربيه:, صنعاء..."
4,– وزير الخارجية السعودي إلى بيروت بين «الخميسي...,no,وزير الخارجية السعودي إلى بيروت بين الخميسين و...,"[وزير, الخارجيه, السعودي, الي, بيروت, بين, الخ..."


In [3]:
import pandas as pd
import re
from google.colab import files

# 🔽 Upload fichier
uploaded = files.upload()

# 🔽 Lire fichier CSV
df = pd.read_csv('nettoye_tokens.csv', encoding='utf-8-sig')
phrases = df['clean_text'].astype(str).tolist()

# Liste personnalisée de stopwords (attention aux espaces et virgules)
custom_stopwords = ['من', 'اى', 'عن', 'الى', 'يوما', 'الآن','مما','حول','ما','نحو', 'حتى', 'ثم', 'على','مهلة', 'الي', 'لا', 'ولا','في', 'لم', 'لن', 'لو', 'لدينا', 'التي', 'بين','وهي','اكثر','مع','هم','لنا','هذا','بعد','بالنسبه','لانها','ايضا','خلال','بعد','هذه','اجل','بكل','ان','ما', 'اولا',
                    'ال', 'او', 'ليس']

custom_stopwords = [normalize_arabic(word) for word in custom_stopwords]#ajouter ce ligne pour lire bien les stop words.

# Tokenisation simple par split sur espace
def simple_tokenize(text):
    return text.split()

# Filtrer les stopwords dans la liste des tokens
def remove_stopwords(tokens):
    return [token for token in tokens if token not in custom_stopwords]


# Traitement complet
tokens_column = []
tokens_filtered_column = []

for phrase in phrases:
    cleaned = normalize_arabic(phrase)
    cleaned = attach_punct_to_previous_word(cleaned)
    tokens = simple_tokenize(cleaned)
    tokens_column.append(tokens)
    filtered_tokens = remove_stopwords(tokens)
    tokens_filtered_column.append(filtered_tokens)

# Ajouter les tokens et les tokens filtrés dans deux colonnes
df['tokens'] = tokens_column
df['tokens_filtered'] = tokens_filtered_column

# Sauvegarder et télécharger
df.to_csv('nettoye_tokens_filtered.csv', index=False, encoding='utf-8-sig')
files.download('nettoye_tokens_filtered.csv')
df.head()

Saving nettoye_tokens.csv to nettoye_tokens (1).csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,النص,yes / no,clean_text,tokens,tokens_filtered
0,– عون: لحكومة تمثل الجميع… مطالب كتل تؤخّر الت...,yes,عون: لحكومة تمثل الجميع مطالب كتل تؤخر التأليف,"[عون:, لحكومه, تمثل, الجميع, مطالب, كتل, توخر,...","[عون:, لحكومه, تمثل, الجميع, مطالب, كتل, توخر,..."
1,– زيارة إيرانية للراعي: رسالة وتطمينات وسجادة ...,no,زيارة إيرانية للراعي: رسالة وتطمينات وسجادة بق...,"[زياره, ايرانيه, للراعي:, رساله, وتطمينات, وسج...","[زياره, ايرانيه, للراعي:, رساله, وتطمينات, وسج..."
2,– تفاؤل بحلّ قضية الودائع: على أي أسس؟!,no,تفاؤل بحل قضية الودائع: على أي أسس,"[تفاول, بحل, قضيه, الودايع:, علي, اي, اسس]","[تفاول, بحل, قضيه, الودايع:, اسس]"
3,– بوادر حملة عسكرية إسرائيلية – غربية: صنعاء ت...,no,بوادر حملة عسكرية إسرائيلية غربية: صنعاء تتحضر...,"[بوادر, حمله, عسكريه, اسراييليه, غربيه:, صنعاء...","[بوادر, حمله, عسكريه, اسراييليه, غربيه:, صنعاء..."
4,– وزير الخارجية السعودي إلى بيروت بين «الخميسي...,no,وزير الخارجية السعودي إلى بيروت بين الخميسين و...,"[وزير, الخارجيه, السعودي, الي, بيروت, بين, الخ...","[وزير, الخارجيه, السعودي, بيروت, الخميسين, وتش..."


In [6]:
# Étape 1 : Installer camel-tools avec tous les modules
!pip install --upgrade camel-tools[all]

# Étape 2 : Télécharger le modèle morphologique arabe
from camel_tools.utils.dl import download_model
download_model('calima-msa-r13')

# Étape 3 : Initialiser l'analyseur
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer

db = MorphologyDB.builtin_db()
analyzer = Analyzer(db)

ModuleNotFoundError: No module named 'camel_tools.utils.dl'

In [2]:
import pandas as pd
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer

# 1. Charger ton fichier existant (avec tokens filtrés)
df = pd.read_csv('nettoye_tokens_filtered.csv', encoding='utf-8-sig')

# 2. Corriger les listes stockées sous forme de chaînes (ast = safe eval)
import ast
df['tokens_filtered'] = df['tokens_filtered'].apply(ast.literal_eval)

# 3. Initialiser l'analyseur CAMeL
db = MorphologyDB.builtin_db()
analyzer = Analyzer(db)

# 4. Appliquer la lemmatisation (LAM) et racinisation (STAM)
lemmatized_tokens = []
stemmed_tokens = []

for tokens in df['tokens_filtered']:
    lemmas = []
    roots = []
    for token in tokens:
        analyses = analyzer.analyze(token)
        if analyses:
            lemmas.append(analyses[0]['lex'])    # Lemma (forme canonique)
            roots.append(analyses[0]['root'])    # Racine
        else:
            lemmas.append(token)  # si non reconnu
            roots.append(token)
    lemmatized_tokens.append(lemmas)
    stemmed_tokens.append(roots)

# 5. Ajouter les résultats au DataFrame
df['lemmes'] = lemmatized_tokens
df['racines'] = stemmed_tokens

# 6. Sauvegarder le fichier final
df.to_csv('corpus_lemmatise_racinise.csv', index=False, encoding='utf-8-sig')

# 7. Télécharger dans Colab
from google.colab import files
files.download('corpus_lemmatise_racinise.csv')

# 8. Afficher un aperçu
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/root/.camel_tools/data/morphology_db/calima-msa-r13/morphology.db'